<a href="https://colab.research.google.com/github/phillip-jaeslee/PULSIM/blob/main/PULSIM_density_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PULSIM — density-matrix pulse sequences

The companion notebook simulates one shaped pulse acting on an isolated spin:
a magnetization vector, an excitation profile. That model cannot describe a
pulse *sequence*, because a sequence needs two things it does not have —
**delays**, during which nothing is irradiated but the spins keep evolving, and
**coupled spins**, so that something can be transferred between them.

This notebook uses the other model: a density matrix propagated through a list
of segments, in Liouville space. Same package, different machinery.

**How to use:** `Runtime` → `Run all`, then edit any cell and re-run it.

| Section | What it does |
|---|---|
| 1. Setup | Clones PULSIM and installs dependencies |
| 2. The model | Spin system, segments, propagation, readout — the whole API |
| 3. Spin echo | Why a delay changes everything; what a 180° does and does not refocus |
| 4. INEPT | Polarization transfer, checked against −sin(2πJΔ) |
| 5. Shaped INEPT | The same sequence with a real shaped pulse instead of an ideal one |
| 6. BIRD | Homonuclear decoupling by a pulse that never touches one of the protons |
| 7. Build your own | Add and remove segments with widgets, no code |

Units throughout: **durations in ms**, **couplings in Hz**, **offsets in rad/ms**
(an offset of *f* kHz is supplied as 2π*f*), **flip angles in radians**.

## 1. Setup

In [ ]:
#@title Install PULSIM
!git clone --quiet --branch main https://github.com/phillip-jaeslee/PULSIM
%cd PULSIM
# The density-matrix path needs only numpy + scipy (the package base). The
# [viz] extra adds matplotlib and ipywidgets, which this notebook uses for
# plotting and for the builder in section 7.
!pip install --quiet ".[viz]"
print("PULSIM ready.")

In [ ]:
#@title Load the simulator  { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt

from PULSIM import RFShape, Pulse, NumpyBackend
from PULSIM.spin_system import SpinSystem, gyro_ratio
from PULSIM.spin_operators import Ix, Iy, Iz, embed, product_operator, SpinOperators
from PULSIM.liouville import Delay, IdealPulse, ShapePulseSegment, LiouvilleSequence
from PULSIM.sequence_figure import draw_sequence

PI = np.pi


def run(segments, spin_system, start="Iz"):
    """Propagate `start` (a product-operator name) through `segments`.

    Returns (final density matrix, SpinOperators for this system) so every
    example below reads out the same way.
    """
    ops = SpinOperators(spin_system)
    sigma0 = ops[start].astype(complex)
    return LiouvilleSequence(segments, spin_system).propagate(sigma0), ops


def table(ops, sigma, names):
    """Print <A> = Tr(sigma A)/Tr(A A) for each named operator."""
    for name, value in ops.readout(sigma, names).items():
        print(f"   {name:<8} {value:+.6f}")

print("Loaded.", len(RFShape.available()), "pulse shapes available.")

## 2. The model

Four objects, and that is the whole API.

**`SpinSystem`** — who the spins are. Nuclei by label, one offset per spin, and
a dict of J-couplings between index pairs.

**Segments** — what happens, in order:

| Segment | Meaning |
|---|---|
| `Delay(duration)` | No RF. Offsets and J-couplings evolve. |
| `IdealPulse(flip, phase, channel)` | A delta-function pulse: RF so strong that nothing else evolves during it. |
| `ShapePulseSegment(pulse)` | A real shaped pulse, sample by sample, with offsets and J evolving throughout. |

**`LiouvilleSequence(segments, spin_system).propagate(sigma)`** — applies
σ → UσU† for every segment in turn.

**`SpinOperators`** — reads the answer back. Spins are named `I`, `S`, `K`, …
in order, so `ops['Iz']` is z on spin 0 and `ops['IzSy']` is the two-spin
product 2·Iz(I)·Iy(S). `ops.readout(sigma, [...])` gives a dict of coefficients.

Run it once on the simplest possible case — a 90° pulse on one spin:

In [ ]:
#@title A single 90 degree pulse, read out as product operators
ss = SpinSystem(nuclei=['H'], offsets=[0.0])

sigma, ops = run([IdealPulse(PI / 2, phase="x", channel='H')], ss, start="Iz")

print("after 90x on equilibrium Iz:")
table(ops, sigma, ['Ix', 'Iy', 'Iz'])
print("\n(Iz -> -Iy is the usual right-handed convention: a +x pulse")
print(" rotates +z toward -y.)")

## 3. Spin echo — what a 180° actually refocuses

A 90° pulse puts magnetization in the transverse plane, where it dephases:
each spin's offset turns it at its own rate, and a J-coupling splits it into
components that turn at different rates. A 180° pulse in the middle of two
equal delays reverses some of that — **but not all of it**, and which part
survives is the single most useful fact in sequence design.

Start from `Ix` on a ¹H coupled to a ¹³C (¹J = 140 Hz), both off resonance, and
compare two echoes of identical length:

- **180° on ¹H only** — inverts one partner of the coupled pair
- **180° on ¹H and ¹³C** — inverts both

Watch `Ix` come back as a function of τ:

In [ ]:
#@title Spin echo: 180 on one channel vs both
J_CH = 140.0        # Hz
OFF_H, OFF_C = 1.7, -0.9    # rad/ms -- deliberately off resonance

def echo(tau, invert_carbon):
    ss = SpinSystem(nuclei=['H', '13C'], offsets=[OFF_H, OFF_C],
                    couplings={(0, 1): J_CH})
    segments = [Delay(tau), IdealPulse(PI, phase="x", channel='H')]
    if invert_carbon:
        segments.append(IdealPulse(PI, phase="x", channel='13C'))
    segments.append(Delay(tau))
    sigma, ops = run(segments, ss, start="Ix")
    return ops.expectation(sigma, 'Ix')

taus = np.linspace(0.0, 2.0 / (J_CH / 1000.0), 200)   # ms
h_only = np.array([echo(t, False) for t in taus])
both   = np.array([echo(t, True)  for t in taus])
theory = np.cos(PI * (J_CH / 1000.0) * 2 * taus)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(taus, h_only, lw=2, label="180 on $^{1}$H only")
ax.plot(taus, both, 'o', ms=3, label="180 on $^{1}$H and $^{13}$C")
ax.plot(taus, theory, '-', lw=1, color='0.4', label=r"theory: $\cos(\pi J \cdot 2\tau)$")
ax.set(xlabel=r"$\tau$ (ms, half-echo)", ylabel="$I_x$ amplitude",
       title=f"Spin echo, J(CH) = {J_CH:.0f} Hz, both spins off resonance")
ax.axhline(0, color='0.85', lw=0.5)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"180 on H only : Ix returns to {h_only.min():.6f} .. {h_only.max():.6f} for every tau")
print(f"180 on both   : max |simulated - theory| = {np.abs(both - theory).max():.2e}")

Both offsets vanish in both cases — that is what an echo is for, and it is why
neither curve depends on `OFF_H` or `OFF_C`. Try changing them.

The difference is the coupling:

- Inverting **one** partner reverses the sign of the J term, so the second delay
  undoes the first. `Ix` comes back at 1.0 for every τ — flat line.
- Inverting **both** leaves the J term unchanged, so it keeps evolving through
  the whole 2τ, giving cos(πJ·2τ).

That asymmetry — *offsets refocus, J does not* — is not a nuisance. It is a
tool, and the next section is what it is for.

## 4. INEPT

INEPT (Morris & Freeman, *JACS* **101**, 760 (1979)) moves polarization from a
sensitive nucleus onto an insensitive one. It is the spin echo above — 180° on
*both* channels, so J keeps evolving — with a 90° pair at the end to convert
the resulting antiphase state into observable coherence on the other spin:

    Iz(I) --90x(I)-- Δ --180x(I), 180x(S)-- Δ --90y(I), 90x(S)-->

The quantity to watch is the coefficient of 2·Iz(I)·Iy(S), which product-operator
algebra says should be exactly −sin(2πJΔ), peaking at Δ = 1/(4J).

In [ ]:
#@title INEPT transfer vs delay
J_HZ = 140.0

def build_inept(Delta, off_I=0.0, off_S=0.0, refocus=True):
    """The sequence as segments. Kept separate from the propagation so the
    figure and the simulation are built from the same object -- the diagram
    is drawn by walking these segments, so it cannot show a different
    sequence from the one that ran."""
    ss = SpinSystem(nuclei=['H', '13C'], offsets=[off_I, off_S],
                    couplings={(0, 1): J_HZ})
    segments = [IdealPulse(PI / 2, phase="x", channel='H'), Delay(Delta)]
    if refocus:
        segments += [IdealPulse(PI, phase="x", channel='H'),
                     IdealPulse(PI, phase="x", channel='13C')]
    segments += [Delay(Delta),
                 IdealPulse(PI / 2, phase="y", channel='H'),
                 IdealPulse(PI / 2, phase="x", channel='13C')]
    return segments, ss

def run_inept(Delta, off_I=0.0, off_S=0.0, refocus=True):
    segments, ss = build_inept(Delta, off_I, off_S, refocus)
    sigma, ops = run(segments, ss, start="Iz")
    return ops.expectation(sigma, 'IzSy')

Delta_opt = 1.0 / (4 * J_HZ / 1000.0)          # ms
deltas = np.linspace(0.0, 2 * Delta_opt, 60)
transfer = np.array([run_inept(d) for d in deltas])
theory = -np.sin(2 * PI * (J_HZ / 1000.0) * deltas)

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(deltas, transfer, 'o', label="simulated")
axs[0].plot(deltas, theory, '-', label=r"theory: $-\sin(2\pi J \Delta)$")
axs[0].axvline(Delta_opt, color='gray', ls=':', label=r"$\Delta = 1/(4J)$")
axs[0].set(xlabel=r"$\Delta$ (ms)", ylabel=r"antiphase S amplitude  $2I_z(I)I_y(S)$",
           title=f"INEPT transfer efficiency (J = {J_HZ:.0f} Hz)")
axs[0].legend(fontsize=9)

segments, ss = build_inept(Delta_opt)
draw_sequence(LiouvilleSequence(segments, ss), ax=axs[1], to_scale=False,
              title=r"ideal INEPT, $\Delta = 1/(4J)$")
fig.tight_layout()
plt.show()

print(f"max |simulated - theory|          : {np.abs(transfer - theory).max():.2e}")
print(f"transfer at Delta = 1/(4J)        : {run_inept(Delta_opt):+.6f}")
print(f"same, far off resonance (refocused): {run_inept(Delta_opt, 2.0, 1.3):+.6f}")
print(f"same, WITHOUT the 180s            : {run_inept(Delta_opt, 2.0, 1.3, refocus=False):+.6f}")

The last two lines are the point of the 180° pair. With it, the transfer is
identical on and off resonance. Without it, an off-resonance spin scrambles
into a mixture of in-phase and antiphase coherence and the transfer collapses.

## 5. The same sequence, with a real pulse

Everything above used `IdealPulse` — infinitely strong, zero duration, perfect
regardless of offset. Replace the two ¹H 90° pulses with a real shaped pulse and
nothing else changes in the code. This is where the two halves of PULSIM meet:
the shape is the same object the companion notebook plots excitation profiles
for.

In [ ]:
#@title Shaped INEPT: eburp1 instead of ideal 90s
SHAPE, SHAPE_MS = "eburp1", 1.5      # try uburp, gausscasq5, hermite, sneeze...

def build_shaped_inept(Delta, off_I=0.0, off_S=0.0):
    ss = SpinSystem(nuclei=['H', '13C'], offsets=[off_I, off_S],
                    couplings={(0, 1): J_HZ})
    shape = RFShape.create(SHAPE, duration=SHAPE_MS, points=1000)
    backend = NumpyBackend(Gamma=gyro_ratio('H'))
    p90x = Pulse(shape, PI / 2, axis="x", backend=backend)
    p90y = Pulse(shape, PI / 2, axis="y", backend=backend)
    return [ShapePulseSegment(p90x),
            Delay(Delta),
            IdealPulse(PI, phase="x", channel='H'),
            IdealPulse(PI, phase="x", channel='13C'),
            Delay(Delta),
            ShapePulseSegment(p90y),
            IdealPulse(PI / 2, phase="x", channel='13C')], ss

def run_shaped(Delta, off_I=0.0, off_S=0.0):
    segments, ss = build_shaped_inept(Delta, off_I, off_S)
    sigma, ops = run(segments, ss, start="Iz")
    return ops.expectation(sigma, 'IzSy')

shaped = np.array([run_shaped(d) for d in deltas])

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(deltas, transfer, '-', lw=2, label="ideal 90s")
axs[0].plot(deltas, shaped, 'o', ms=4, label=f"{SHAPE}, {SHAPE_MS} ms")
axs[0].axvline(Delta_opt, color='gray', ls=':')
axs[0].set(xlabel=r"$\Delta$ (ms)", ylabel=r"$2I_z(I)I_y(S)$",
           title="INEPT with ideal vs shaped excitation")
axs[0].legend(fontsize=9)

segments, ss = build_shaped_inept(Delta_opt)
draw_sequence(LiouvilleSequence(segments, ss), ax=axs[1], to_scale=False,
              title=f"shaped INEPT ({SHAPE}, {SHAPE_MS} ms)")
fig.tight_layout()
plt.show()

print(f"ideal  at Delta = 1/(4J): {run_inept(Delta_opt):+.6f}")
print(f"shaped at Delta = 1/(4J): {run_shaped(Delta_opt):+.6f}")
print("\nThe shaped pulse has a finite duration, during which J and the offsets")
print("keep evolving -- so the curve is shifted, not merely scaled. That shift is")
print("the thing a sequence designer has to compensate for, and it is invisible")
print("to an excitation-profile calculation.")

## 6. BIRD

BIRD (Garbow, Weitekamp & Pines, *Chem. Phys. Lett.* **93**, 504 (1982)) does
something that sounds impossible: it inverts one proton and not another, using
only non-selective ¹H pulses that hit both.

Three spins — a proton **I** bonded to ¹³C (¹J ≈ 140 Hz), a second proton **I′**
not bonded to it, and the carbon **S**. I and I′ are indistinguishable to any ¹H
pulse. The only thing that differs is whether they are coupled to S, and the
simultaneous 180° on the carbon channel is what turns that difference into a
different net rotation.

    90x(H) — τ — [180x(H), 180x(S)] — τ — 90±x(H)      τ = 1/(2·¹J(CH))

In [ ]:
#@title BIRD as a coupling-selective filter
J_CH_3, J_HH = 140.0, 7.0

def bird_system():
    return SpinSystem(nuclei=['H', 'H', '13C'], offsets=[0.0, 0.0, 0.0],
                      couplings={(0, 2): J_CH_3, (0, 1): J_HH})

def bird_cluster(final_phase="x"):
    tau = 1.0 / (2 * J_CH_3 / 1000.0)
    return [IdealPulse(PI / 2, phase="x", channel='H'),
            Delay(tau),
            IdealPulse(PI, phase="x", channel='H'),
            IdealPulse(PI, phase="x", channel='13C'),
            Delay(tau),
            IdealPulse(PI / 2, phase=final_phase, channel='H')]

print("BIRD as a filter -- each proton started alone at +z:")
print(f"   {'final 90':<12} {'I (bonded to S)':>18} {'I-prime (not bonded)':>22}")
for phase, label in [("x", "90x ... 90x"), ("-x", "90x ... 90-x")]:
    segs = bird_cluster(phase)
    sig_I,  ops = run(segs, bird_system(), start="Iz")
    sig_Ip, _   = run(segs, bird_system(), start="Sz")
    print(f"   {label:<12} {ops.expectation(sig_I, 'Iz'):>18.4f} "
          f"{ops.expectation(sig_Ip, 'Sz'):>22.4f}")
print("\n(Spin 1 is the second proton I-prime, which SpinOperators labels S;")
print(" the carbon is spin 2, labelled K.)")

The bonded proton inverts, the other comes back untouched — and swapping the
final pulse's phase swaps which one.

Now use that as a decoupler. Put BIRD where the plain 180° normally sits in a
homonuclear echo: a plain 180° inverts both protons, which leaves their mutual
J(HH) completely unrefocused, while BIRD inverts only one of them, which
refocuses it.

There is a catch, and it is worth seeing rather than being told. The BIRD
cluster contains a 180° on the carbon, and **it leaves the carbon inverted**.
Over the outer echo that means both partners of the I–S pair have been flipped,
so J(CH) no longer refocuses either — and J(CH) is 140 Hz against J(HH)'s 7 Hz,
so it dominates. Restoring the carbon with one more 180° on that channel fixes
it. Both are plotted below.

In [ ]:
#@title BIRD as a homonuclear decoupler
def outer_echo(tau_outer, middle):
    segs = [Delay(tau_outer)] + middle + [Delay(tau_outer)]
    sigma, ops = run(segs, bird_system(), start="Ix")
    return ops.expectation(sigma, 'Ix')

tau_bird = 1.0 / (2 * J_CH_3 / 1000.0)
toll = np.cos(PI * (J_HH / 1000.0) * 2 * tau_bird)   # BIRD's own internal cost

# Does the cluster leave the carbon where it found it?
_sig, _ops = run(bird_cluster("x"), bird_system(), start="Kz")
print(f"BIRD cluster: Kz(13C) -> {_ops.expectation(_sig, 'Kz'):+.4f}"
      "   (inverted -- see below)")

plain_180   = [IdealPulse(PI, phase="x", channel='H')]
bird_raw    = bird_cluster("x")
bird_restore = bird_cluster("x") + [IdealPulse(PI, phase="x", channel='13C')]

# -- panel A: one full J(HH) period, the decoupling claim ------------------
outer = np.linspace(0.0, 1000.0 / J_HH, 300)          # ms
plain = np.array([outer_echo(t, plain_180) for t in outer])
restored = np.array([outer_echo(t, bird_restore) for t in outer])
theory_plain = np.cos(2 * PI * (J_HH / 1000.0) * outer)

# -- panel B: a 15 ms zoom, two periods of 1/J(CH) -------------------------
zoom = np.linspace(0.0, 15.0, 200)
raw_z = np.array([outer_echo(t, bird_raw) for t in zoom])
res_z = np.array([outer_echo(t, bird_restore) for t in zoom])
theory_raw = -toll * np.cos(PI * (J_CH_3 / 1000.0) * 2 * zoom)

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(outer, plain, lw=2, label="plain 180 in the middle")
axs[0].plot(outer, theory_plain, '--', lw=1, color='0.4',
            label=r"theory: $\cos(2\pi J_{HH}\tau')$")
axs[0].plot(outer, restored, lw=2, color='C3', label="BIRD + 180($^{13}$C)")
axs[0].axhline(-toll, color='C3', ls=':', label="BIRD's fixed toll")
axs[0].set(xlabel=r"$\tau'$ (ms)", ylabel="$I_x(I)$ amplitude",
           title=f"Decoupling I from I' (J(HH) = {J_HH:.0f} Hz)")
axs[0].legend(fontsize=9)

axs[1].plot(zoom, raw_z, lw=2, color='C1', label="BIRD as written (carbon left inverted)")
axs[1].plot(zoom, theory_raw, '--', lw=1, color='0.4',
            label=r"$-\mathrm{toll}\cdot\cos(\pi J_{CH}\cdot 2\tau')$")
axs[1].plot(zoom, res_z, lw=2, color='C3', label="BIRD + 180($^{13}$C)")
axs[1].set(xlabel=r"$\tau'$ (ms, zoom)", ylabel="$I_x(I)$ amplitude",
           title=f"Why the carbon has to be restored (J(CH) = {J_CH_3:.0f} Hz)")
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"\nplain 180  vs cos(2 pi J_HH tau')  : max diff {np.abs(plain - theory_plain).max():.2e}")
print(f"plain 180  amplitude range         : [{plain.min():+.4f}, {plain.max():+.4f}]  (modulated)")
print(f"BIRD as written, std over the zoom : {raw_z.std():.4f}  (J(CH)-modulated, NOT decoupled)")
print(f"   matches -toll*cos(pi J_CH 2tau') : max diff {np.abs(raw_z - theory_raw).max():.2e}")
print(f"BIRD + 180(13C), std over the zoom : {res_z.std():.2e}  (flat)")
print(f"BIRD + 180(13C), std over 1 J_HH period: {restored.std():.2e}  (flat -> decoupled)")
print(f"BIRD's fixed toll                  : {-toll:+.4f}  = -cos(pi*J_HH*2*tau_BIRD)")

**Top panel** — a plain 180° inverts both protons together, so their mutual
coupling is never refocused and the signal is fully modulated by cos(2πJ_HH·τ′).
BIRD with the carbon restored inverts only the bonded proton, refocuses the pair,
and comes back flat. The small constant offset is J(HH) evolving unrefocused
during BIRD's own two internal delays — a fixed toll, not a τ′-dependent one.

**Bottom panel** — why the extra carbon 180° is not optional. Without it the
cluster leaves ¹³C inverted, so J(CH) is unrefocused across the outer echo and
the signal oscillates at 140 Hz, twenty times faster than the coupling we were
trying to remove. It follows −toll·cos(πJ_CH·2τ′) exactly. A coarse τ′ grid can
easily step over whole periods of this and make the curve *look* flat, which is
worth remembering whenever a simulated sweep looks too clean.

## 7. Build your own

Everything above is a list of segments. This builds one with widgets instead —
add pulses and delays, set the spin system, press Run. No fixed length.

In [ ]:
#@title Sequence builder  { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

NUCLEI = ["H", "D", "T", "13C", "15N", "19F", "31P"]
SEG_KINDS = ["ideal pulse", "delay", "shaped pulse"]
PHASES = ["x", "y", "-x", "-y"]

def _w(px): return widgets.Layout(width=px)
def _lbl(t, px): return widgets.Label(t, layout=_w(px))

# ---- shapes that can actually be simulated (see the companion notebook) ----
def _usable_shapes():
    backend = NumpyBackend(Gamma=gyro_ratio("H"))
    out = []
    for name in sorted(RFShape.available()):
        if name in {"file", "composite"}:
            continue
        try:
            s = RFShape.create(name, duration=1.0, points=64)
            p = (Pulse(s, backend=backend) if s.calibration_mode == "adiabatic"
                 else Pulse(s, flip=PI / 2, backend=backend))
            p.calibrated_rf()
            out.append(name)
        except Exception:
            pass
    return out

SHAPES = _usable_shapes()

# ---- spin system ----------------------------------------------------------
w_nspins = widgets.IntSlider(value=2, min=1, max=3, description="spins:",
                             style={"description_width": "initial"}, layout=_w("200px"))
spin_box, coupling_box = widgets.VBox([]), widgets.VBox([])
_spins, _couplings = [], []

def _rebuild_system(change=None):
    n = w_nspins.value
    while len(_spins) < n:
        i = len(_spins)
        _spins.append({
            "nucleus": widgets.Dropdown(options=NUCLEI, value="H" if i == 0 else "13C",
                                        layout=_w("90px")),
            "offset": widgets.FloatText(value=0.0, layout=_w("90px"))})
    del _spins[n:]
    spin_box.children = tuple(
        widgets.HBox([_lbl(f"spin {i} ({'ISK'[i]})", "90px"), s["nucleus"], s["offset"]])
        for i, s in enumerate(_spins))

    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    old = {p["pair"]: p["J"].value for p in _couplings}
    _couplings.clear()
    for (i, j) in pairs:
        _couplings.append({"pair": (i, j),
                           "J": widgets.FloatText(value=old.get((i, j), 0.0), layout=_w("90px"))})
    coupling_box.children = tuple(
        widgets.HBox([_lbl(f"J({i},{j}) Hz", "90px"), c["J"]])
        for c, (i, j) in zip(_couplings, pairs))

w_nspins.observe(_rebuild_system, names="value")

# ---- segment rows ---------------------------------------------------------
_rows = []
seg_box = widgets.VBox([])
out = widgets.Output()

def _refresh():
    for n, r in enumerate(_rows, start=1):
        r["idx"].value = str(n)
    seg_box.children = tuple(r["box"] for r in _rows)

def _make_row(kind="ideal pulse", flip=90.0, phase="x", channel="H",
              duration=1.0, shape="gausscasq5", points=500):
    idx = _lbl("", "26px")
    w_kind = widgets.Dropdown(options=SEG_KINDS, value=kind, layout=_w("120px"))
    w_flip = widgets.FloatText(value=flip, layout=_w("75px"))
    w_ph   = widgets.Dropdown(options=PHASES, value=phase, layout=_w("60px"))
    w_ch   = widgets.Dropdown(options=NUCLEI, value=channel, layout=_w("80px"))
    w_dur  = widgets.FloatText(value=duration, layout=_w("80px"))
    w_shp  = widgets.Dropdown(options=SHAPES, value=shape if shape in SHAPES else SHAPES[0],
                              layout=_w("130px"))
    w_pts  = widgets.IntText(value=points, layout=_w("75px"))
    w_rm   = widgets.Button(description="\u2715", tooltip="remove", layout=_w("36px"))

    def _sync(change=None):
        k = w_kind.value
        w_flip.disabled = (k == "delay")
        w_ph.disabled   = (k == "delay")
        w_ch.disabled   = (k == "delay")
        w_shp.disabled  = (k != "shaped pulse")
        w_pts.disabled  = (k != "shaped pulse")
        w_dur.disabled  = (k == "ideal pulse")   # ideal pulse duration cancels
    w_kind.observe(_sync, names="value")
    _sync()

    def _remove(_b):
        if len(_rows) == 1:
            with out:
                clear_output(wait=True)
                print("A sequence needs at least one segment.")
            return
        _rows.remove(row)
        _refresh()
    w_rm.on_click(_remove)

    row = {"idx": idx, "kind": w_kind, "flip": w_flip, "phase": w_ph, "channel": w_ch,
           "duration": w_dur, "shape": w_shp, "points": w_pts,
           "box": widgets.HBox([idx, w_kind, w_flip, w_ph, w_ch, w_dur, w_shp, w_pts, w_rm])}
    return row

def _add(_b=None):
    if _rows:
        p = _rows[-1]
        _rows.append(_make_row(p["kind"].value, p["flip"].value, p["phase"].value,
                               p["channel"].value, p["duration"].value,
                               p["shape"].value, p["points"].value))
    else:
        _rows.append(_make_row())
    _refresh()

# ---- run ------------------------------------------------------------------
w_start = widgets.Text(value="Iz", description="start:", style={"description_width": "initial"},
                       layout=_w("160px"))
w_read  = widgets.Text(value="Ix, Iy, Iz, IzSy", description="read out:",
                       style={"description_width": "initial"}, layout=_w("320px"))

def _build():
    nuclei  = [s["nucleus"].value for s in _spins]
    offsets = [s["offset"].value for s in _spins]
    couplings = {c["pair"]: c["J"].value for c in _couplings if c["J"].value != 0.0}
    ss = SpinSystem(nuclei=nuclei, offsets=offsets, couplings=couplings)

    segments = []
    for r in _rows:
        k = r["kind"].value
        if k == "delay":
            segments.append(Delay(r["duration"].value))
        elif k == "ideal pulse":
            segments.append(IdealPulse(np.deg2rad(r["flip"].value),
                                       phase=r["phase"].value,
                                       channel=r["channel"].value))
        else:
            shape = RFShape.create(r["shape"].value, duration=r["duration"].value,
                                   points=int(r["points"].value))
            backend = NumpyBackend(Gamma=gyro_ratio(r["channel"].value))
            segments.append(ShapePulseSegment(
                Pulse(shape, np.deg2rad(r["flip"].value), axis=r["phase"].value,
                      backend=backend)))
    return segments, ss

def _run(_b):
    with out:
        clear_output(wait=True)
        try:
            segments, ss = _build()
            sigma, ops = run(segments, ss, start=w_start.value.strip())
            names = [n.strip() for n in w_read.value.split(",") if n.strip()]
            print(f"{len(segments)} segments on {', '.join(ss.nuclei)}"
                  f"   start = {w_start.value.strip()}")
            table(ops, sigma, names)
            fig, ax = plt.subplots(figsize=(7, 2.6))
            draw_sequence(LiouvilleSequence(segments, ss), ax=ax, to_scale=False,
                          title="your sequence")
            fig.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")

b_add = widgets.Button(description="+ Add segment", layout=_w("130px"))
b_run = widgets.Button(description="Run", button_style="primary", layout=_w("110px"))
b_add.on_click(_add)
b_run.on_click(_run)

# ---- default: the INEPT of section 4 --------------------------------------
_rebuild_system()
_spins[0]["nucleus"].value, _spins[1]["nucleus"].value = "H", "13C"
_couplings[0]["J"].value = 140.0
_Delta = 1.0 / (4 * 140.0 / 1000.0)
for _a in [("ideal pulse", 90.0, "x", "H", 0.0),
           ("delay", 0.0, "x", "H", _Delta),
           ("ideal pulse", 180.0, "x", "H", 0.0),
           ("ideal pulse", 180.0, "x", "13C", 0.0),
           ("delay", 0.0, "x", "H", _Delta),
           ("ideal pulse", 90.0, "y", "H", 0.0),
           ("ideal pulse", 90.0, "x", "13C", 0.0)]:
    _rows.append(_make_row(_a[0], _a[1], _a[2], _a[3], _a[4]))
_refresh()

header = widgets.HBox([_lbl("#", "26px"), _lbl("segment", "120px"), _lbl("flip \u00b0", "75px"),
                       _lbl("phase", "60px"), _lbl("channel", "80px"), _lbl("dur ms", "80px"),
                       _lbl("shape", "130px"), _lbl("points", "75px")])

display(widgets.VBox([
    widgets.HTML("<b>Spin system</b>  &mdash; offsets in rad/ms, couplings in Hz"),
    w_nspins, spin_box, coupling_box,
    widgets.HTML("<b>Segments</b>  &mdash; applied in order"),
    header, seg_box,
    widgets.HBox([b_add, b_run]),
    widgets.HTML("<b>Readout</b>  &mdash; product-operator names, comma separated"),
    widgets.HBox([w_start, w_read]),
    out,
]))

---

### Known limitations of this version

- **No relaxation.** Propagation is unitary: T1 and T2 are ignored everywhere.
  Every sequence here would decay in a real experiment, and long delays decay most.
- **No phase cycling.** Not needed to read out a coherence here — `SpinOperators`
  projects onto whatever operator you ask for — but it means these simulations
  do not reproduce what a spectrometer's receiver actually sums.
- **No gradients or diffusion.**
- **Ideal pulses are genuinely ideal.** `IdealPulse` omits offset and coupling
  terms entirely, so it never suffers off-resonance error. Only `ShapePulseSegment`
  shows what a real pulse does — section 5 is the honest comparison.
- **Cost grows as 4ⁿ.** Each segment propagates a 2ⁿ × 2ⁿ density matrix, and a
  shaped pulse does it once per sample. Three spins is comfortable in Colab;
  beyond about five, expect to wait.
- **Offsets are rad/ms, couplings are Hz.** An asymmetry inherited from the
  package's time base. An offset of *f* kHz is entered as 2π*f*.

### Citing

If PULSIM is useful in your work, please cite the repository:
<https://github.com/phillip-jaeslee/PULSIM>